# Brady-GPT, live

Type a question, or paste a class moment, and hear what my model would say.

**To start:** use a GPU (a T4 works, at 8-bit), add `HF_TOKEN` under the key icon (Secrets), and choose **Runtime > Run all**. Allow about 10 minutes the first time, so start before class.

**To replay a class:** on my Mac, run `uv run python demo/pick_moment.py` to list classes, `demo/pick_moment.py 12 --show` to number a class's turns, and `demo/pick_moment.py 12 --at 40` to copy everything said before turn 40. Then paste it below.

Nothing run here is saved with this notebook. Don't save a copy after pasting class text.

In [ ]:
%pip install -q transformers==5.21.0 accelerate bitsandbytes

In [ ]:
import random, time

import torch
import transformers
from google.colab import userdata
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextStreamer

transformers.logging.set_verbosity_error()
MINE = "bradyrutherford/minerva-olmo3-7b-B"
login_token = userdata.get("HF_TOKEN")  # my model is private: this login is the only way to load it
HfApi(token=login_token).model_info(MINE)
print("HF_TOKEN: can read my private model")

# Full precision needs about 15 GB, which is all a T4 has, so a T4 gets 8-bit and bigger GPUs get the real thing.
full = torch.cuda.get_device_properties(0).total_memory / 1e9 >= 20
settings = {"dtype": torch.bfloat16} if full else {
    "dtype": torch.float16, "quantization_config": BitsAndBytesConfig(load_in_8bit=True)}
start = time.time()
tok = AutoTokenizer.from_pretrained(MINE, token=login_token)
model = AutoModelForCausalLM.from_pretrained(MINE, device_map={"": 0}, token=login_token, **settings).eval()
print(f"Ready: {torch.cuda.get_device_name(0)}, {'full precision' if full else '8-bit'}, loaded in {time.time() - start:.0f}s")

STOPS = ["\nProfessor:", "\nStudent ", "\nSpeaker:", "\nBrady:"]  # it may never speak as someone else


@torch.inference_mode()
def say(text, class_name="CS156 Finding Patterns in Data with Machine Learning - Class presentations"):
    """A full moment from pick_moment.py goes in as it is. Anything else counts as the professor's question."""
    text = text.strip()
    if not text.startswith("Answer as:"):
        text = f"Answer as: Brady\nClass discussion in {class_name}\n\nProfessor: {text}\n\nWhat do you say?"
    ids = tok(f"<|user|>\n{text}\n<|assistant|>\n", return_tensors="pt", add_special_tokens=False).to(model.device)
    torch.manual_seed(random.randrange(10**6))
    start = time.time()
    out = model.generate(**ids, do_sample=True, temperature=0.8, top_p=0.95, max_new_tokens=256,
                         eos_token_id=tok.eos_token_id, stop_strings=STOPS, tokenizer=tok,
                         streamer=TextStreamer(tok, skip_prompt=True, skip_special_tokens=True))
    print(f"\n\n[{ids.input_ids.shape[1]:,} tokens read, {time.time() - start:.0f}s]")

## Ask it

Replace the text between the quotes, then press Shift+Enter. Run the cell again for a different answer.

In [ ]:
say("""
If this model can sound like me, what's still missing before it could really join a class discussion?
""")

## After class

Tick `wipe` and run this cell. It deletes the session, including the private model and anything pasted here.

In [ ]:
wipe = False  #@param {type:"boolean"}
if wipe:
    from google.colab import runtime
    runtime.unassign()